### Execution of Tensor Train Decompositions on Color Video (4D Tensor)
This notebook performs the core tensor decomposition experiments on the high-order (4D) RGB color video sequence. It executes the proposed Online TT-ALS algorithm alongside three algebraic baselines: TT-FOA, Batch TT-ALS (Full), and Batch TT-ALS (Slice). The reconstructed tensors and computational times are saved locally to be evaluated by the subsequent metric calculation notebook.

In [ ]:
# ---------------------------------------------------------
# Environment Setup and Initialization
# ---------------------------------------------------------
using LinearAlgebra
using Random
using TensorToolbox
using Plots
using Images
using ImageIO
using FileIO
using FFMPEG_jll
using Printf
using Statistics
using Revise
using JLD2

# Explicitly restrict BLAS to a single thread.
# This ensures a strictly fair and reproducible evaluation of the computational execution time 
# across all algorithms, as described in the experimental setup of the paper.
BLAS.set_num_threads(1)

In [ ]:
# Include custom modules containing the proposed method and baselines
includet("../julia_source/myGenerateData.jl")
includet("../julia_source/myLoadVideo.jl")
includet("../julia_source/myTTD.jl")

### Dataset Preparation & Loading (Color Video)

Due to file size limits and licensing, the original video datasets are not included in this repository. 
To reproduce our experiments, please download the "continuousPan" sequence from the [CDnet 2014 benchmark](http://www.changedetection.net/).

**Note for custom video sequences:**
If you wish to evaluate the algorithms on your own custom color video data, prepare a folder containing the sequential frame images and update the folder path in the `load_color_video("path/to/your/custom/video")` function below.

In [ ]:
# ---------------------------------------------------------
# Dataset Loading and Preprocessing (Color Video)
# ---------------------------------------------------------
# Load the color video sequence which serves as the ground-truth streaming tensor.
# The resulting tensor is 4D: (Height, Width, Channels(RGB), Time).
video_tensor, h, w, T = load_color_video("../.cache/kagglehub/datasets/maamri95/cdnet2014/versions/1/dataset/PTZ/intermittentPan/input")

([0.9725490196078431 0.9725490196078431 … 0.24705882352941178 0.2235294117647059; 0.9764705882352941 0.9764705882352941 … 0.2235294117647059 0.21176470588235294; … ; 0.4745098039215686 0.4745098039215686 … 0.7176470588235294 0.6980392156862745; 0.4823529411764706 0.4823529411764706 … 0.6666666666666666 0.6313725490196078;;; 0.8823529411764706 0.8823529411764706 … 0.20784313725490197 0.1843137254901961; 0.8862745098039215 0.8862745098039215 … 0.1843137254901961 0.17254901960784313; … ; 0.403921568627451 0.403921568627451 … 0.5254901960784314 0.5058823529411764; 0.4117647058823529 0.4117647058823529 … 0.4745098039215686 0.4392156862745098;;; 0.8196078431372549 0.8196078431372549 … 0.1607843137254902 0.13725490196078433; 0.8235294117647058 0.8156862745098039 … 0.13725490196078433 0.12549019607843137; … ; 0.403921568627451 0.403921568627451 … 0.4588235294117647 0.4392156862745098; 0.4117647058823529 0.4117647058823529 … 0.40784313725490196 0.37254901960784315;;;; 0.9137254901960784 0.93725

In [19]:
T = 500;
# video_tensor = video_tensor[:, :, :, 251:250+T];
video_tensor = video_tensor[:, :, :, 1:T];
ttsizes = [h, w, 3, T];
ttranks = [30, 3, 3];

@show ttsizes;

ttsizes = [480, 704, 3, 500]


In [8]:
save_color_video(video_tensor, "Original", [0], "results/color/pedestrians", "pedestrians_original.mp4")

[ Info: Saved animation to /home/takeda/12/results/color/pedestrians/pedestrians_original.mp4


動画を保存しました: results/color/pedestrians/pedestrians_original.mp4


In [ ]:
# ---------------------------------------------------------
# 1. Proposed Method: Online TT-ALS
# ---------------------------------------------------------

# Execute the exact, single-sweep online TT decomposition with incremental orthogonalization
G_history_est, gn_history_est, iter_time = online_ttALS(video_tensor, ttsizes, ttranks)

video_tensor_est = Array{Float64, 4}(undef, h, w, 3, T)
relative_error = zeros(T)
for t in 1:T
    video_tensor_est[:, :, :, t] = ttProduct(G_history_est[t], gn_history_est[t])
    relative_error[t] = norm(video_tensor[:, :, :, t] - video_tensor_est[:, :, :, t]) / norm(video_tensor[:, :, :, t])
end
@printf("Average Relative Error: %.6f\n", mean(relative_error))

Average Relative Error: 0.084215


In [ ]:
# ---------------------------------------------------------
# 2. Baseline Method: TT-FOA (First-Order Approximation)
# ---------------------------------------------------------

# Execute the recursive online method based on first-order approximations.
# lambda: Forgetting factor for Recursive Least Squares (RLS).
# rho: Regularization parameter.
G_history_FOA_est, gn_history_FOA_est, iter_time_FOA = ttFOA(video_tensor, ttsizes, ttranks, 0.7)

video_tensor_FOA_est = Array{Float64, 4}(undef, h, w, 3, T)
relative_error_FOA = zeros(T)
for t in 1:T
    video_tensor_FOA_est[:, :, :, t] = ttProduct(G_history_FOA_est[t], gn_history_FOA_est[t])
    relative_error_FOA[t] = norm(video_tensor[:, :, :, t] - video_tensor_FOA_est[:, :, :, t]) / norm(video_tensor[:, :, :, t])
end
@printf("Average Relative Error (FOA): %.6f\n", mean(relative_error_FOA))

Average Relative Error (FOA): 0.129022


In [ ]:
# ---------------------------------------------------------
# 3. Baseline Method: Batch TT-ALS (Full Sequence)
# ---------------------------------------------------------

# Apply standard TT-ALS directly to the entire 3D streaming tensor.
G_batch_est, iter_time_batch = @timed batch_ttALS(video_tensor, ttranks, max_sweeps=10, tol=1e-10)
iter_time_batch *=1e3  # in milliseconds
video_tensor_batch_est_full = tt2full(G_batch_est)
relative_error_batch_full = norm(video_tensor - video_tensor_batch_est_full) / norm(video_tensor)
@printf("Relative Error (Batch TT-ALS): %.6f\n", relative_error_batch_full)

Relative Error (Batch TT-ALS): 0.203410


In [ ]:
# ---------------------------------------------------------
# 4. Baseline Method: Batch TT-ALS (Slice-by-Slice)
# ---------------------------------------------------------

# Apply Batch TT-ALS sequentially to each 2D data slice independently.
iter_time_batchSlice = zeros(T)
G_batchSlice_est = Vector{Vector{Array{Float64}}}(undef, T)

for t in 1:T
    video_tensor_t = video_tensor[:, :, :, t]
    ttranks_t = [ttranks[1], 3]
    rlt = @timed batch_ttALS(video_tensor_t, ttranks_t, max_sweeps=10, tol=1e-10)
    G_batchSlice_est[t] = rlt.value
    iter_time_batchSlice[t] = rlt.time*1e3  # in milliseconds
end

video_tensor_batchSlice_est = Array{Float64, 4}(undef, h, w, 3, T)
relative_error_batchSlice = zeros(T)
for t in 1:T
    video_tensor_batchSlice_est[:, :, :, t] = tt2full(G_batchSlice_est[t])
    relative_error_batchSlice[t] = norm(video_tensor[:, :, :, t] - video_tensor_batchSlice_est[:, :, :, t]) / norm(video_tensor[:, :, :, t])
end
@printf("Average Relative Error (Batch Slice): %.6f\n", mean(relative_error_batchSlice))

Average Relative Error (Batch Slice): 0.111098


In [ ]:
# ---------------------------------------------------------
# Save Experimental Results
# ---------------------------------------------------------
# Serialize the reconstructed 4D video tensors and computational execution times to a JLD2 file.
# These intermediate results will be loaded by the evaluation notebook (ALS_indicator_color.ipynb) 
# to compute the mathematical, structural, and perceptual quality metrics independently.

@save "data_color_continuousPan_$(ttranks[1]).jld2" video_tensor_est video_tensor_FOA_est video_tensor_batch_est_full video_tensor_batchSlice_est iter_time iter_time_FOA iter_time_batch iter_time_batchSlice